In [1]:
import os
import pandas as pd
import numpy as np
import joblib

#### Constants

In [2]:
str_bucket = 'assessment-alt'
str_task = '03_preprocessing'
str_target = 'price'
str_dirname_output = './output'
os.makedirs(str_dirname_output, exist_ok=True)
print(f'Bucket: {str_bucket}')
print(f'Task: {str_task}')

Bucket: assessment-alt
Task: 03_preprocessing


#### Load Data

In [3]:
%%time
df_train = pd.read_csv(f's3://{str_bucket}/02_data_split/train.csv')
df_valid = pd.read_csv(f's3://{str_bucket}/02_data_split/valid.csv')
df_test = pd.read_csv(f's3://{str_bucket}/02_data_split/test.csv')
print(f'Train: {df_train.shape}')
print(f'Valid: {df_valid.shape}')
print(f'Test:  {df_test.shape}')

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


Train: (62407, 9)
Valid: (21535, 9)
Test:  (16058, 9)
CPU times: user 409 ms, sys: 45.6 ms, total: 454 ms
Wall time: 734 ms


#### TextNormalizer

In [4]:
class TextNormalizer:
    def __init__(self):
        pass

    def fit(self, X):
        print('TextNormalizer fit...')
        return self

    def transform(self, X):
        print('TextNormalizer transform...')
        X = X.copy()
        X['subject'] = X['subject'].str.lower().str.strip()
        X['brand'] = X['brand'].str.lower().str.strip()
        X['variety'] = X['variety'].str.lower().str.strip().fillna('base')
        X['card_number'] = X['card_number'].astype(str).str.lower().str.strip()
        X['grading_company'] = X['grading_company'].str.lower().str.strip()
        return X

#### CardKeyFeatures

Creates a card key from the fungibility tuple and computes card-level statistics from training data. This is a **stateful** transformer — `fit()` learns price statistics from training data.

In [5]:
class CardKeyFeatures:
    def __init__(self):
        self.df_card_stats = None

    def fit(self, X):
        print('CardKeyFeatures fit...')
        X = X.copy()
        X['str_card_key'] = (
            X['year'].astype(str) + '|' +
            X['subject'] + '|' +
            X['brand'] + '|' +
            X['variety'] + '|' +
            X['card_number'] + '|' +
            X['grade'].astype(str) + '|' +
            X['grading_company']
        )
        self.df_card_stats = X.groupby('str_card_key')['price'].agg(
            int_card_txn_count='count',
            flt_card_median_price='median',
            flt_card_mean_price='mean'
        ).reset_index()
        print(f'  Learned stats for {len(self.df_card_stats):,} unique cards')
        return self

    def transform(self, X):
        print('CardKeyFeatures transform...')
        X = X.copy()
        X['str_card_key'] = (
            X['year'].astype(str) + '|' +
            X['subject'] + '|' +
            X['brand'] + '|' +
            X['variety'] + '|' +
            X['card_number'] + '|' +
            X['grade'].astype(str) + '|' +
            X['grading_company']
        )
        X = X.merge(self.df_card_stats, on='str_card_key', how='left')
        X['int_card_txn_count'] = X['int_card_txn_count'].fillna(0).astype(int)
        X['flt_card_median_price'] = X['flt_card_median_price'].fillna(0)
        X['flt_card_mean_price'] = X['flt_card_mean_price'].fillna(0)
        return X

#### SubjectFeatures

Computes subject-level price statistics from training data. **Stateful**.

In [6]:
class SubjectFeatures:
    def __init__(self):
        self.df_subject_stats = None

    def fit(self, X):
        print('SubjectFeatures fit...')
        self.df_subject_stats = X.groupby('subject')['price'].agg(
            int_subject_txn_count='count',
            flt_subject_median_price='median'
        ).reset_index()
        print(f'  Learned stats for {len(self.df_subject_stats):,} unique subjects')
        return self

    def transform(self, X):
        print('SubjectFeatures transform...')
        X = X.copy()
        X = X.merge(self.df_subject_stats, on='subject', how='left')
        X['int_subject_txn_count'] = X['int_subject_txn_count'].fillna(0).astype(int)
        X['flt_subject_median_price'] = X['flt_subject_median_price'].fillna(0)
        return X

#### BrandFeatures

Computes brand-level price statistics from training data. **Stateful**.

In [7]:
class BrandFeatures:
    def __init__(self):
        self.df_brand_stats = None

    def fit(self, X):
        print('BrandFeatures fit...')
        self.df_brand_stats = X.groupby('brand')['price'].agg(
            int_brand_txn_count='count',
            flt_brand_median_price='median'
        ).reset_index()
        print(f'  Learned stats for {len(self.df_brand_stats):,} unique brands')
        return self

    def transform(self, X):
        print('BrandFeatures transform...')
        X = X.copy()
        X = X.merge(self.df_brand_stats, on='brand', how='left')
        X['int_brand_txn_count'] = X['int_brand_txn_count'].fillna(0).astype(int)
        X['flt_brand_median_price'] = X['flt_brand_median_price'].fillna(0)
        return X

#### GradeFeatures

Creates grade-related features. **Stateless**.

In [8]:
class GradeFeatures:
    def __init__(self):
        pass

    def fit(self, X):
        print('GradeFeatures fit...')
        return self

    def transform(self, X):
        print('GradeFeatures transform...')
        X = X.copy()
        X['flt_grade'] = X['grade'].astype(float)
        X['int_is_psa'] = (X['grading_company'] == 'psa').astype(int)
        X['int_is_gem_mint'] = (X['grade'] == 10.0).astype(int)
        return X

#### TimeFeatures

Creates time-based features from the transaction date. **Stateless**.

In [9]:
class TimeFeatures:
    def __init__(self):
        self.dt_first_txn = pd.Timestamp('2018-05-16')

    def fit(self, X):
        print('TimeFeatures fit...')
        return self

    def transform(self, X):
        print('TimeFeatures transform...')
        X = X.copy()
        X['date'] = pd.to_datetime(X['date'])
        X['int_year_sold'] = X['date'].dt.year
        X['int_month_sold'] = X['date'].dt.month
        X['int_days_since_first_txn'] = (X['date'] - self.dt_first_txn).dt.days
        return X

#### TargetTransform

Creates log-transformed price target. **Stateless**.

In [10]:
class TargetTransform:
    def __init__(self):
        pass

    def fit(self, X):
        print('TargetTransform fit...')
        return self

    def transform(self, X):
        print('TargetTransform transform...')
        X = X.copy()
        X['flt_log_price'] = np.log1p(X['price'])
        return X

#### PrepareFeatures

Selects the final feature set for modeling. **Stateful** — stores the feature column list.

In [11]:
class PrepareFeatures:
    def __init__(self):
        self.list_str_features = [
            'flt_grade', 'int_is_psa', 'int_is_gem_mint',
            'int_card_txn_count', 'flt_card_median_price', 'flt_card_mean_price',
            'int_subject_txn_count', 'flt_subject_median_price',
            'int_brand_txn_count', 'flt_brand_median_price',
            'int_year_sold', 'int_month_sold', 'int_days_since_first_txn',
        ]

    def fit(self, X):
        print('PrepareFeatures fit...')
        print(f'  Feature columns: {len(self.list_str_features)}')
        return self

    def transform(self, X):
        print('PrepareFeatures transform...')
        X = X.copy()
        list_str_keep = self.list_str_features + ['flt_log_price']
        X = X[list_str_keep]
        return X

#### Preprocessing Model

In [12]:
class PreprocessingModel:
    def __init__(self, list_cls_transformers):
        self.list_cls_transformers = list_cls_transformers

    def transform(self, X):
        for cls_transformer in self.list_cls_transformers:
            X = cls_transformer.transform(X)
        return X

#### Fit Transformers

In [13]:
%%time
# fit each transformer on training data
cls_text_normalizer = TextNormalizer().fit(df_train)
df_tmp = cls_text_normalizer.transform(df_train)

cls_card_key_features = CardKeyFeatures().fit(df_tmp)
df_tmp = cls_card_key_features.transform(df_tmp)

cls_subject_features = SubjectFeatures().fit(df_tmp)
df_tmp = cls_subject_features.transform(df_tmp)

cls_brand_features = BrandFeatures().fit(df_tmp)
df_tmp = cls_brand_features.transform(df_tmp)

cls_grade_features = GradeFeatures().fit(df_tmp)
df_tmp = cls_grade_features.transform(df_tmp)

cls_time_features = TimeFeatures().fit(df_tmp)
df_tmp = cls_time_features.transform(df_tmp)

cls_target_transform = TargetTransform().fit(df_tmp)
df_tmp = cls_target_transform.transform(df_tmp)

cls_prepare_features = PrepareFeatures().fit(df_tmp)
df_tmp = cls_prepare_features.transform(df_tmp)

print(f'\nTrain shape after pipeline: {df_tmp.shape}')
df_tmp.head()

TextNormalizer fit...
TextNormalizer transform...
CardKeyFeatures fit...
  Learned stats for 15,269 unique cards
CardKeyFeatures transform...
SubjectFeatures fit...
  Learned stats for 333 unique subjects
SubjectFeatures transform...
BrandFeatures fit...
  Learned stats for 390 unique brands
BrandFeatures transform...
GradeFeatures fit...
GradeFeatures transform...
TimeFeatures fit...
TimeFeatures transform...
TargetTransform fit...
TargetTransform transform...
PrepareFeatures fit...
  Feature columns: 13
PrepareFeatures transform...

Train shape after pipeline: (62407, 14)
CPU times: user 529 ms, sys: 43.5 ms, total: 573 ms
Wall time: 571 ms


,flt_grade,int_is_psa,int_is_gem_mint,int_card_txn_count,flt_card_median_price,flt_card_mean_price,int_subject_txn_count,flt_subject_median_price,int_brand_txn_count,flt_brand_median_price,int_year_sold,int_month_sold,int_days_since_first_txn,flt_log_price
0,10.0,1,1,1,700.00,700.00,136,50.50,4,689.995,2018,5,0,6.552508
1,9.0,1,0,2,739.99,739.99,136,50.50,4,689.995,2018,7,61,6.685848
2,9.0,1,0,2,739.99,739.99,136,50.50,4,689.995,2018,7,70,6.523548
3,9.5,0,0,1,56.00,56.00,1,56.00,9,56.000,2018,8,84,4.043051
4,9.5,0,0,1,41.26,41.26,4,40.63,9,56.000,2018,8,90,3.743841


#### Build Pipeline

In [14]:
# collect fit transformers into list
list_cls_transformers = [
    cls_text_normalizer,
    cls_card_key_features,
    cls_subject_features,
    cls_brand_features,
    cls_grade_features,
    cls_time_features,
    cls_target_transform,
    cls_prepare_features,
]
print(f'Pipeline has {len(list_cls_transformers)} transformers')

Pipeline has 8 transformers


#### Initialize Preprocessing Model

In [15]:
cls_model_preprocessing = PreprocessingModel(list_cls_transformers)

#### Save Preprocessing Model

In [16]:
joblib.dump(cls_model_preprocessing, f'{str_dirname_output}/cls_model_preprocessing.joblib')
print(f'Saved preprocessing_model.joblib to {str_dirname_output}')

Saved preprocessing_model.joblib to ./output


#### Transform All Splits

In [17]:
%%time

df_train_processed = cls_model_preprocessing.transform(df_train)
print(f'Train: {df_train_processed.shape}')

df_valid_processed = cls_model_preprocessing.transform(df_valid)
print(f'Valid: {df_valid_processed.shape}')

df_test_processed = cls_model_preprocessing.transform(df_test)
print(f'Test:  {df_test_processed.shape}')

TextNormalizer transform...
CardKeyFeatures transform...
SubjectFeatures transform...
BrandFeatures transform...
GradeFeatures transform...
TimeFeatures transform...
TargetTransform transform...
PrepareFeatures transform...
Train: (62407, 14)
TextNormalizer transform...
CardKeyFeatures transform...
SubjectFeatures transform...
BrandFeatures transform...
GradeFeatures transform...
TimeFeatures transform...
TargetTransform transform...
PrepareFeatures transform...
Valid: (21535, 14)
TextNormalizer transform...
CardKeyFeatures transform...
SubjectFeatures transform...
BrandFeatures transform...
GradeFeatures transform...
TimeFeatures transform...
TargetTransform transform...
PrepareFeatures transform...
Test:  (16058, 14)
CPU times: user 636 ms, sys: 40.1 ms, total: 676 ms
Wall time: 670 ms


#### X, y Split

In [18]:
# Separate features and target
str_target_col = 'flt_log_price'
list_str_features = [c for c in df_train_processed.columns if c != str_target_col]

X_train = df_train_processed[list_str_features]
y_train = df_train_processed[[str_target_col]]

X_valid = df_valid_processed[list_str_features]
y_valid = df_valid_processed[[str_target_col]]

X_test = df_test_processed[list_str_features]
y_test = df_test_processed[[str_target_col]]

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_valid: {X_valid.shape}, y_valid: {y_valid.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')
print(f'\nFeatures: {list_str_features}')

X_train: (62407, 13), y_train: (62407, 1)
X_valid: (21535, 13), y_valid: (21535, 1)
X_test:  (16058, 13), y_test:  (16058, 1)

Features: ['flt_grade', 'int_is_psa', 'int_is_gem_mint', 'int_card_txn_count', 'flt_card_median_price', 'flt_card_mean_price', 'int_subject_txn_count', 'flt_subject_median_price', 'int_brand_txn_count', 'flt_brand_median_price', 'int_year_sold', 'int_month_sold', 'int_days_since_first_txn']


#### Save preprocessed data to s3

In [19]:
%%time

for str_name, df_x, df_y in [('train', X_train, y_train), ('valid', X_valid, y_valid), ('test', X_test, y_test)]:
    str_s3_path_x = f's3://{str_bucket}/{str_task}/X_{str_name}.csv'
    str_s3_path_y = f's3://{str_bucket}/{str_task}/y_{str_name}.csv'
    df_x.to_csv(str_s3_path_x, index=False)
    df_y.to_csv(str_s3_path_y, index=False)
    print(f'Saved X_{str_name}.csv ({df_x.shape}) and y_{str_name}.csv ({df_y.shape}) to S3')

Saved X_train.csv ((62407, 13)) and y_train.csv ((62407, 1)) to S3
Saved X_valid.csv ((21535, 13)) and y_valid.csv ((21535, 1)) to S3
Saved X_test.csv ((16058, 13)) and y_test.csv ((16058, 1)) to S3
CPU times: user 758 ms, sys: 3.9 ms, total: 762 ms
Wall time: 1.26 s


#### Takeaways

- **8 transformers** in the pipeline: text normalization, card/subject/brand stats, grade features, time features, target transform, and feature selection
- **Stateful transformers** (CardKeyFeatures, SubjectFeatures, BrandFeatures) learn statistics from training data only — preventing data leakage
- **13 numeric features** produced for modeling
- **Log-transformed price** (flt_log_price) used as target to handle the heavy right skew
- Cards not seen in training get 0 for card-level stats — the model must learn to handle these cold start cases
- Preprocessing model saved as joblib for reproducible inference